# Otsu 大津法自动阈值图像分割实验：学生练习版

本实验实现一个基于 **Otsu 大津法** 的自动阈值图像分割程序。练习版保留图像生成、网络图像读取、结果显示和实验分析框架，将关键算法步骤设置为 `TODO`，需要学生补全。

需要完成的核心内容：

1. 手写灰度直方图统计。
2. 手写阈值二值分割。
3. 手写全局 Otsu 阈值计算。
4. 手写局部块 Otsu 分割。
5. 手写 Sobel 梯度幅值计算。
6. 手写平均梯度加权直方图。
7. 基于给定直方图重新实现 Otsu 阈值计算。

## 1. Otsu 大津法相关知识

### 1.1 阈值图像分割

灰度图像可以表示为：

$$
I(x,y) \in [0,255]
$$

阈值分割使用一个阈值 `T` 将图像划分为前景和背景：

$$
B(x,y)=
\begin{cases}
1, & I(x,y) \ge T\\
0, & I(x,y) < T
\end{cases}
$$

如果前景比背景更暗，也可以使用相反规则：

$$
B(x,y)=
\begin{cases}
1, & I(x,y) < T\\
0, & I(x,y) \ge T
\end{cases}
$$

因此，阈值分割不仅要选择阈值，还要明确“亮区域为前景”还是“暗区域为前景”。

### 1.2 灰度直方图

灰度直方图统计图像中每个灰度值出现的次数：

$$
h(k)=\#\{(x,y)\mid I(x,y)=k\}
$$

其中 `k = 0, 1, ..., 255`。

如果图像的前景和背景灰度差异明显，直方图通常会出现两个较明显的峰值。理想阈值往往位于两个峰之间，使两类像素尽可能分离。

### 1.3 Otsu 大津法的核心思想

Otsu 方法会枚举所有候选阈值 `T`，并选择让前景和背景 **类间方差最大** 的阈值。

给定阈值 `T` 后，图像像素被分为两类：

- 类 0：灰度值 `0...T`
- 类 1：灰度值 `T+1...255`

设：

- $\omega_0(T)$：类 0 的像素比例
- $\omega_1(T)$：类 1 的像素比例
- $\mu_0(T)$：类 0 的平均灰度
- $\mu_1(T)$：类 1 的平均灰度

类间方差为：

$$
\sigma_b^2(T)=\omega_0(T)\omega_1(T)(\mu_0(T)-\mu_1(T))^2
$$

Otsu 最佳阈值为：

$$
T^*=\arg\max_T \sigma_b^2(T)
$$

直观理解：如果某个阈值能让两类像素的平均灰度差异最大，并且两类像素都占有一定比例，那么该阈值通常能较好地区分前景和背景。

### 1.4 全局 Otsu 与局部 Otsu

**全局 Otsu**：整幅图像只计算一个阈值。

优点：

1. 实现简单。
2. 计算速度快。
3. 当前景和背景灰度分布明显分离时效果好。

局限：

1. 光照不均时，一个阈值可能无法适应所有区域。
2. 局部区域灰度差异明显时，全局阈值容易漏分或误分。

**局部 Otsu**：将图像分成多个局部块，每个块单独计算 Otsu 阈值。

优点：

1. 能适应局部光照变化。
2. 对局部灰度差异明显的图像更灵活。

局限：

1. 分块边界可能产生块状伪影。
2. 小块中如果前景/背景样本不足，阈值可能不稳定。
3. 分块大小需要根据图像内容调整。

### 1.5 计算步骤总结

全局 Otsu：

```text
输入：灰度图像 image
步骤 1：统计灰度直方图 hist
步骤 2：将 hist 转换为概率分布 p
步骤 3：枚举阈值 T = 0...255
步骤 4：计算每个 T 下的 w0, w1, mu0, mu1
步骤 5：计算类间方差 sigma_b^2(T)
步骤 6：选择类间方差最大的 T
步骤 7：根据 T 对整幅图像进行二值化
输出：最佳阈值 T 与分割结果
```

局部 Otsu：

```text
输入：灰度图像 image，块大小 block_size
步骤 1：将图像按 block_size 划分为多个小块
步骤 2：对每个小块单独计算 Otsu 阈值
步骤 3：用该小块阈值分割该小块
步骤 4：将所有小块分割结果拼接成完整图像
输出：局部阈值图 threshold_map 与局部分割结果
```

## 学生练习任务与代码补全步骤

请按照下面顺序补全代码。建议不要跳着写，因为后面的实验会依赖前面函数的输出。

### 2.1 补全顺序

1. **补全 `compute_histogram`**
   - 将输入灰度图像展开为一维数组。
   - 创建长度为 `bins` 的计数数组。
   - 遍历每个像素灰度值，并让对应灰度级计数加 1。
   - 返回灰度直方图。

2. **补全 `threshold_segment`**
   - 判断 `foreground` 是 `"bright"` 还是 `"dark"`。
   - 如果前景是亮目标，则令 `image >= threshold` 的像素为前景。
   - 如果前景是暗目标，则令 `image < threshold` 的像素为前景。
   - 返回 `0/1` 二值图像。

3. **补全 `otsu_threshold`**
   - 使用 `compute_histogram` 得到灰度直方图。
   - 将直方图归一化为概率分布。
   - 枚举候选阈值 `T = 0, 1, ..., 255`。
   - 对每个阈值计算背景类和前景类的权重 `w0, w1`。
   - 计算两类均值 `mu0, mu1`。
   - 计算类间方差 `w0 * w1 * (mu0 - mu1) ** 2`。
   - 保存每个阈值的类间方差，并返回类间方差最大的阈值。

4. **补全 `local_block_otsu`**
   - 根据 `block_size` 遍历图像中的每个局部块。
   - 对每个局部块调用 `otsu_threshold` 得到局部阈值。
   - 对该局部块调用 `threshold_segment` 得到局部分割结果。
   - 将局部分割结果写回整幅二值图。
   - 将局部阈值写入 `threshold_map`。
   - 记录每个块的位置和阈值。

5. **补全 `compute_gradient_magnitude`**
   - 使用边缘填充处理图像边界。
   - 使用 Sobel 水平方向模板计算 `gx`。
   - 使用 Sobel 垂直方向模板计算 `gy`。
   - 计算梯度幅值 `sqrt(gx ** 2 + gy ** 2)`。

6. **补全 `compute_gradient_weighted_histogram`**
   - 先调用 `compute_gradient_magnitude` 得到梯度幅值图。
   - 按灰度级统计梯度幅值之和。
   - 按灰度级统计像素数量。
   - 对每个出现过的灰度级计算平均梯度幅值。
   - 返回平均梯度加权直方图和梯度幅值图。

7. **补全 `otsu_threshold_from_histogram`**
   - 输入不再是图像，而是已经构造好的直方图。
   - 将直方图归一化为概率分布。
   - 按照 Otsu 的类间方差最大化准则枚举阈值。
   - 返回最佳阈值和每个阈值对应的类间方差。

### 2.2 检查建议

1. 每补完一个函数，先运行该函数所在单元。
2. 全局 Otsu 能运行后，再继续补局部 Otsu。
3. 局部 Otsu 能显示阈值图后，再继续补直方图变换法。
4. 最后运行完整 notebook，比较直接 Otsu 与直方图修正后 Otsu 的结果差异。

## 2. 导入基础库

本实验使用：

- `numpy`：基础数组计算。
- `matplotlib`：图像显示。

Otsu 阈值计算、直方图统计、全局分割和局部分块分割均手写实现。

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 3. 准备经典灰度测试图像与光照不均图像

本实验不再从网络下载图片，而是在程序中构造两类灰度测试图像：

1. **经典硬币/圆形目标灰度测试图像**：前景目标较亮、背景较暗，适合观察全局 Otsu 的基本计算过程。
2. **光照不均测试图像**：整幅图像属于同一背景区域，但从左到右、从上到下存在明显亮度变化，使相同背景在不同位置呈现出很大的灰度差异。该图像用于观察局部 Otsu 相比全局 Otsu 的优势。

这样可以先理解 Otsu 的基本原理，再重点观察局部 Otsu 对光照不均图像的适应能力。

In [ ]:
def make_classic_coin_image(height=220, width=300, random_state=3):
    """
    构造一张经典硬币灰度测试图像。

    参数：
        height, width: 图像高度和宽度
        random_state: 随机种子，用于生成可复现实验噪声

    返回：
        image: uint8 类型灰度图像
    """
    rng = np.random.default_rng(random_state)
    yy, xx = np.mgrid[0:height, 0:width]

    image = np.full((height, width), 55, dtype=np.float64)
    image += 12 * np.sin(xx / 28) + 8 * np.cos(yy / 35)

    coins = [
        (78, 72, 34, 178),
        (150, 92, 42, 205),
        (228, 75, 31, 168),
        (95, 158, 38, 190),
        (205, 155, 45, 215),
    ]

    for cx, cy, radius, value in coins:
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2
        highlight = 18 * np.exp(-((xx - cx + 10) ** 2 + (yy - cy + 10) ** 2) / (2 * (radius / 2) ** 2))
        image[mask] = value + highlight[mask]

    image += rng.normal(0, 8, size=image.shape)
    return np.clip(image, 0, 255).astype(np.uint8)


def make_uneven_illumination_image(height=240, width=360, random_state=11):
    """
    构造一张更明显的光照不均灰度图像。

    这张图像模拟一张带暗色标记的纸面：纸面本身属于同一背景区域，
    但左下角处在阴影中，右上角受到强光照射，因此背景灰度在不同位置
    差异很大。图像中的前景目标是暗色笔画和圆形标记，适合观察全局
    Otsu 在光照不均时的误分，以及局部 Otsu 的自适应阈值效果。

    参数：
        height, width: 图像高度和宽度
        random_state: 随机种子，用于生成可复现实验噪声

    返回：
        image: uint8 类型灰度图像
    """
    rng = np.random.default_rng(random_state)
    yy, xx = np.mgrid[0:height, 0:width]

    # 同一纸面背景：整体从左暗到右亮，并叠加局部阴影和右上强光。
    horizontal_light = 70 + 135 * (xx / (width - 1))
    vertical_light = 16 * (1 - yy / (height - 1))
    left_bottom_shadow = -58 * np.exp(-((xx - 58) ** 2 / (2 * 95 ** 2) + (yy - 196) ** 2 / (2 * 70 ** 2)))
    right_top_spotlight = 42 * np.exp(-((xx - 296) ** 2 / (2 * 78 ** 2) + (yy - 44) ** 2 / (2 * 52 ** 2)))
    paper_texture = 5 * np.sin(xx / 23) + 4 * np.cos((xx + yy) / 31)
    background = horizontal_light + vertical_light + left_bottom_shadow + right_top_spotlight + paper_texture

    image = background.copy()

    # 前景目标：用多个暗色笔画组成清晰目标，分布在暗区、中间区和亮区。
    foreground_mask = np.zeros((height, width), dtype=bool)

    # 左侧暗区的矩形目标
    foreground_mask[45:70, 38:128] = True
    foreground_mask[78:98, 58:145] = True
    foreground_mask[108:132, 42:118] = True

    # 中部椭圆目标
    ellipse = ((xx - 178) / 55) ** 2 + ((yy - 122) / 36) ** 2 <= 1
    foreground_mask |= ellipse

    # 右侧亮区的条形目标
    foreground_mask[54:80, 242:330] = True
    foreground_mask[96:118, 232:318] = True
    foreground_mask[145:169, 224:342] = True

    # 底部跨越不同光照区域的长条目标
    foreground_mask[190:212, 72:292] = True

    # 暗色目标跟随局部光照变化：在亮区仍然更亮，在暗区更暗，因此全局阈值更难处理。
    image[foreground_mask] = background[foreground_mask] - 58

    # 给前景边缘加一点自然变化，避免完全机械的块状灰度。
    image[foreground_mask] += rng.normal(0, 3, size=foreground_mask.sum())
    image += rng.normal(0, 3.5, size=image.shape)

    return np.clip(image, 0, 255).astype(np.uint8)


raw_image = make_classic_coin_image()
gray_image = raw_image.copy()

uneven_image = make_uneven_illumination_image()

print("经典硬币图像形状：", gray_image.shape)
print("光照不均图像形状：", uneven_image.shape)
print("光照不均图像灰度范围：", int(uneven_image.min()), "到", int(uneven_image.max()))

## 4. 图像显示

下面先显示经典硬币图像和光照不均图像。请重点观察光照不均图像中背景区域的亮度变化：左侧背景偏暗，右侧背景偏亮，同一背景类别在不同位置出现了较大的灰度差异。

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap="gray", vmin=0, vmax=255)
plt.title("经典硬币灰度图像")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(uneven_image, cmap="gray", vmin=0, vmax=255)
plt.title("光照不均灰度图像")
plt.axis("off")

plt.tight_layout()
plt.show()

## 5. 实现灰度直方图与阈值分割

In [ ]:
def compute_histogram(image, bins=256):
    """
    TODO：手写灰度直方图统计。

    参数：
        image: 灰度图像，通常为 uint8 类型，灰度范围为 0 到 255
        bins: 灰度级数量，默认 256

    返回：
        hist: 每个灰度值出现的次数，长度为 bins
    """
    # TODO 1：创建长度为 bins 的直方图数组。
    # TODO 2：将二维图像展开为一维像素序列。
    # TODO 3：遍历每个像素值，并让对应灰度级计数加 1。
    # TODO 4：返回 hist。
    raise NotImplementedError("请补全 compute_histogram 函数")


def threshold_segment(image, threshold, foreground="bright"):
    """
    TODO：根据阈值进行二值分割。

    参数：
        image: 灰度图像
        threshold: 分割阈值
        foreground: "bright" 表示亮区域为前景，"dark" 表示暗区域为前景

    返回：
        binary: 二值分割结果，前景为 1，背景为 0
    """
    # TODO 1：如果 foreground == "bright"，将 image >= threshold 的位置置为 1。
    # TODO 2：如果 foreground == "dark"，将 image < threshold 的位置置为 1。
    # TODO 3：返回 uint8 类型的 0/1 二值图像。
    # TODO 4：如果 foreground 不是上述两种取值，抛出 ValueError。
    raise NotImplementedError("请补全 threshold_segment 函数")


def plot_histogram_with_threshold(image, threshold=None, title="灰度直方图"):
    """
    显示灰度直方图，并标出阈值位置。
    """
    hist = compute_histogram(image)
    gray_levels = np.arange(256)

    plt.bar(gray_levels, hist, width=1.0, color="#9aa6b2", alpha=0.85)
    if threshold is not None:
        plt.axvline(threshold, color="#dc2626", linewidth=2.5, label=f"阈值 T={threshold}")
        plt.legend()
    plt.xlabel("灰度值")
    plt.ylabel("像素数量")
    plt.title(title)

## 6. 实现 Otsu 大津法

下面根据类间方差最大原则计算最佳阈值。

In [ ]:
def otsu_threshold(image):
    """
    TODO：手写 Otsu 自动阈值计算。

    参数：
        image: 灰度图像

    返回：
        best_threshold: 最佳阈值
        between_vars: 每个候选阈值对应的类间方差
    """
    # TODO 1：调用 compute_histogram 统计灰度直方图。
    # TODO 2：计算总像素数，并将直方图转换为概率分布。
    # TODO 3：创建 gray_levels = [0, 1, ..., 255]。
    # TODO 4：初始化 best_threshold、best_var 和 between_vars。
    # TODO 5：枚举阈值 T，分别计算两类权重 w0、w1。
    # TODO 6：跳过 w0 或 w1 为 0 的无效阈值。
    # TODO 7：计算两类均值 mu0、mu1。
    # TODO 8：计算类间方差，并更新最佳阈值。
    # TODO 9：返回最佳阈值和 between_vars。
    raise NotImplementedError("请补全 otsu_threshold 函数")


def plot_between_class_variance(between_vars, best_threshold):
    """
    显示不同候选阈值对应的类间方差。
    """
    plt.plot(np.arange(256), between_vars, color="#159947")
    plt.axvline(best_threshold, color="#dc2626", linewidth=2, label=f"最佳阈值 T={best_threshold}")
    plt.xlabel("候选阈值 T")
    plt.ylabel("类间方差")
    plt.title("Otsu 类间方差曲线")
    plt.grid(alpha=0.3)
    plt.legend()

## 7. 全局 Otsu 阈值分割

整幅图像只计算一个 Otsu 阈值，并用该阈值完成二值分割。

In [ ]:
foreground_mode = "bright"

global_T, between_vars = otsu_threshold(gray_image)
global_binary = threshold_segment(gray_image, global_T, foreground=foreground_mode)

plt.figure(figsize=(14, 8))

plt.subplot(2, 3, 1)
plt.imshow(gray_image, cmap="gray", vmin=0, vmax=255)
plt.title("灰度图像")
plt.axis("off")

plt.subplot(2, 3, 2)
plot_histogram_with_threshold(gray_image, global_T, title="灰度直方图与 Otsu 阈值")

plt.subplot(2, 3, 3)
plot_between_class_variance(between_vars, global_T)

plt.subplot(2, 3, 4)
plt.imshow(global_binary, cmap="gray", vmin=0, vmax=1)
plt.title(f"全局 Otsu 分割结果 T={global_T}")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(gray_image, cmap="gray", vmin=0, vmax=255)
plt.imshow(global_binary, cmap="Greens", alpha=0.35, vmin=0, vmax=1)
plt.title("全局分割前景叠加")
plt.axis("off")

plt.tight_layout()
plt.show()

print(f"全局 Otsu 最佳阈值：{global_T}")

## 8. 局部像素块 Otsu 分割：光照不均图像

下面将 **光照不均图像** 划分成若干块，在每个块中单独应用 Otsu 方法计算阈值。

为什么要使用光照不均图像？

1. 全局 Otsu 对整幅图像只使用一个阈值，当背景亮度从左到右变化很大时，单一阈值可能把较暗背景误判为前景。
2. 局部 Otsu 在每个局部块内单独估计阈值，可以根据当前位置的局部亮度变化调整分割标准。
3. 块越小，越能适应局部灰度变化，但阈值可能不稳定；块越大，结果越接近全局阈值。

In [ ]:
def local_block_otsu(image, block_size=64, foreground="bright"):
    """
    TODO：基于局部像素块的 Otsu 分割。

    参数：
        image: 输入灰度图像
        block_size: 局部块大小，例如 32、64、96
        foreground: 前景方向，"bright" 表示亮目标，"dark" 表示暗目标

    返回：
        local_binary: 局部分块 Otsu 分割结果
        threshold_map: 每个像素对应的局部阈值
        block_thresholds: 每个图像块的位置和阈值信息
    """
    h, w = image.shape
    local_binary = np.zeros((h, w), dtype=np.uint8)
    threshold_map = np.zeros((h, w), dtype=np.float64)
    block_thresholds = []

    # TODO 1：按照 block_size 遍历图像块左上角坐标 y0、x0。
    # TODO 2：计算当前块的右下边界 y1、x1，注意不能超过图像尺寸。
    # TODO 3：取出当前局部块 block。
    # TODO 4：调用 otsu_threshold(block) 得到该块阈值 T。
    # TODO 5：调用 threshold_segment(block, T, foreground) 得到局部分割结果。
    # TODO 6：把局部分割结果写入 local_binary 对应区域。
    # TODO 7：把当前块阈值 T 写入 threshold_map 对应区域。
    # TODO 8：将 (x0, y0, x1, y1, T) 加入 block_thresholds。
    # TODO 9：返回 local_binary、threshold_map、block_thresholds。
    raise NotImplementedError("请补全 local_block_otsu 函数")


local_foreground_mode = "dark"

# 为了观察局部 Otsu 的作用，先在同一张光照不均图像上计算全局 Otsu 作为对照。
uneven_global_T, uneven_between_vars = otsu_threshold(uneven_image)
uneven_global_binary = threshold_segment(
    uneven_image,
    uneven_global_T,
    foreground=local_foreground_mode,
)

block_size = 64
local_binary, threshold_map, block_thresholds = local_block_otsu(
    uneven_image,
    block_size=block_size,
    foreground=local_foreground_mode,
)

print(f"光照不均图像的全局 Otsu 阈值：{uneven_global_T}")
print(f"局部 Otsu 分块大小：{block_size} × {block_size}")
print("前 8 个局部块阈值：")
for item in block_thresholds[:8]:
    x0, y0, x1, y1, T = item
    print(f"块区域 x[{x0}:{x1}], y[{y0}:{y1}] -> 阈值 {T}")

## 9. 光照不均图像的局部 Otsu 分割结果与阈值图

下面比较同一张光照不均图像上的全局 Otsu 和局部 Otsu。重点观察：

1. 全局阈值是否会把暗背景区域误判为前景。
2. 局部阈值图是否随图像不同位置的光照变化而变化。
3. 局部分割结果是否比全局分割更能适应背景灰度变化。

In [ ]:
plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(uneven_image, cmap="gray", vmin=0, vmax=255)
plt.title("光照不均原图")
plt.axis("off")

plt.subplot(2, 3, 2)
plot_histogram_with_threshold(
    uneven_image,
    uneven_global_T,
    title=f"灰度直方图与全局阈值 T={uneven_global_T}",
)

plt.subplot(2, 3, 3)
plt.imshow(uneven_global_binary, cmap="gray", vmin=0, vmax=1)
plt.title("全局 Otsu 分割")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(local_binary, cmap="gray", vmin=0, vmax=1)
plt.title(f"局部 Otsu 分割 block={block_size}")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(threshold_map, cmap="viridis")
plt.colorbar(label="局部阈值")
plt.title("局部阈值图")
plt.axis("off")

plt.subplot(2, 3, 6)
difference = np.abs(local_binary.astype(np.int16) - uneven_global_binary.astype(np.int16))
plt.imshow(difference, cmap="gray", vmin=0, vmax=1)
plt.title("局部结果与全局结果差异")
plt.axis("off")

plt.tight_layout()
plt.show()

## 10. 不同块大小对光照不均图像局部 Otsu 的影响

观察不同 `block_size` 下，局部 Otsu 的分割效果和阈值变化。块大小是局部阈值分割中的重要参数：

1. 较小块可以更灵活地适应光照变化，但可能受到噪声和局部目标比例影响。
2. 较大块更稳定，但对光照不均的适应能力会下降。

In [ ]:
block_sizes = [32, 48, 64, 96]

plt.figure(figsize=(14, 7))

for i, bs in enumerate(block_sizes):
    binary_bs, threshold_map_bs, _ = local_block_otsu(
        uneven_image,
        block_size=bs,
        foreground=local_foreground_mode,
    )

    plt.subplot(2, len(block_sizes), i + 1)
    plt.imshow(binary_bs, cmap="gray", vmin=0, vmax=1)
    plt.title(f"局部 Otsu\nblock={bs}")
    plt.axis("off")

    plt.subplot(2, len(block_sizes), i + 1 + len(block_sizes))
    plt.imshow(threshold_map_bs, cmap="viridis")
    plt.title(f"阈值图\nblock={bs}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 11. 直方图变换法修正直方图后再进行 Otsu 分割

前面的实验直接使用原始灰度直方图计算 Otsu 阈值。当图像存在光照不均、背景灰度范围较宽或前景背景灰度重叠时，原始直方图中的谷点可能不明显，直接使用 Otsu 得到的阈值不一定理想。

**直方图变换法** 的基本思想是：利用像素的某种局部性质，将原来的直方图变换成更有利于阈值检测的直方图。例如，可以让原本不明显的波谷变得更深，或者让某些边界相关的灰度级形成更明显的峰值，从而使谷点或峰点更容易被检测。

由微分算子的性质可知：

1. 目标内部和背景内部通常比较平坦，像素梯度较小。
2. 目标与背景之间的边界灰度变化明显，像素梯度较大。
3. 因此，可以根据像素梯度值或灰度级的平均梯度构造 **梯度加权直方图**。

在本实验中，灰度级 `g` 的梯度加权直方图使用该灰度级像素的 **平均梯度幅值**：

$$
H_w(g)=\frac{1}{N_g}\sum_{(x,y), I(x,y)=g} |\nabla I(x,y)|
$$

其中，`N_g` 表示灰度值为 `g` 的像素数量。也就是：对于灰度值为 `g` 的所有像素，不再只统计像素个数，而是计算这些像素位置的平均梯度幅值。这样，位于目标边界附近、灰度变化更明显的灰度级会获得更大的权重。

本节不再使用前面程序生成的两张图像，而是读取一张 **网络灰度图像**，并比较两种方法：

1. 直接对网络图像的原始直方图使用 Otsu。
2. 先用平均梯度加权直方图修正直方图，再对修正后的直方图使用 Otsu。

In [ ]:
def compute_gradient_magnitude(image):
    """
    TODO：使用 Sobel 微分算子计算灰度图像的梯度幅值。

    参数：
        image: 输入灰度图像

    返回：
        gradient: 每个像素位置的梯度幅值
    """
    # TODO 1：将 image 转换为 float64，避免计算溢出。
    # TODO 2：使用 np.pad 对图像边界进行填充。
    # TODO 3：按照 Sobel x 方向模板计算 gx。
    # TODO 4：按照 Sobel y 方向模板计算 gy。
    # TODO 5：计算 gradient = sqrt(gx ** 2 + gy ** 2)。
    # TODO 6：返回 gradient。
    raise NotImplementedError("请补全 compute_gradient_magnitude 函数")


def compute_gradient_weighted_histogram(image):
    """
    TODO：根据灰度级的平均梯度幅值构造梯度加权直方图。

    参数：
        image: 输入灰度图像

    返回：
        weighted_hist: 平均梯度加权直方图
        gradient: 梯度幅值图
    """
    # TODO 1：调用 compute_gradient_magnitude(image) 得到 gradient。
    # TODO 2：创建 gradient_sum，用于统计每个灰度级的梯度幅值之和。
    # TODO 3：创建 gray_count，用于统计每个灰度级的像素数量。
    # TODO 4：遍历图像像素及其梯度值，更新 gradient_sum 和 gray_count。
    # TODO 5：对出现过的灰度级计算 average_gradient = gradient_sum / gray_count。
    # TODO 6：返回 weighted_hist 和 gradient。
    raise NotImplementedError("请补全 compute_gradient_weighted_histogram 函数")


def otsu_threshold_from_histogram(hist):
    """
    TODO：根据给定直方图计算 Otsu 阈值。

    这里的 hist 不一定是普通像素数量直方图，也可以是梯度加权直方图。

    参数：
        hist: 长度为 256 的非负直方图

    返回：
        best_threshold: Otsu 最佳阈值
        between_vars: 每个候选阈值对应的类间方差
    """
    # TODO 1：将 hist 转换为 float64。
    # TODO 2：计算 total，如果 total <= 0，则返回 0 和全 0 的 between_vars。
    # TODO 3：将 hist 归一化为概率 prob。
    # TODO 4：计算全局灰度均值 global_mean。
    # TODO 5：枚举 threshold，逐步计算前景/背景权重与均值。
    # TODO 6：计算类间方差，并保存到 between_vars。
    # TODO 7：记录类间方差最大的 threshold。
    # TODO 8：返回 best_threshold 和 between_vars。
    raise NotImplementedError("请补全 otsu_threshold_from_histogram 函数")


def normalize_for_display(image):
    """
    将图像归一化到 0 到 1，便于显示梯度图。
    """
    image = image.astype(np.float64)
    image_min = image.min()
    image_max = image.max()
    if image_max == image_min:
        return np.zeros_like(image)
    return (image - image_min) / (image_max - image_min)

In [ ]:
from io import BytesIO
from urllib.request import urlopen

from PIL import Image


NETWORK_IMAGE_URL = "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/coins.png"


def load_grayscale_image_from_url(url):
    """
    从网络地址读取图像，并转换为灰度图像。

    参数：
        url: 网络图像地址

    返回：
        image: uint8 类型灰度图像
    """
    with urlopen(url, timeout=20) as response:
        image_bytes = response.read()
    image = Image.open(BytesIO(image_bytes)).convert("L")
    return np.array(image, dtype=np.uint8)


def compare_direct_and_transformed_otsu(image, title, foreground="bright"):
    """
    比较直接 Otsu 与直方图变换后 Otsu 的分割效果。

    参数：
        image: 输入灰度图像
        title: 图像名称
        foreground: 前景方向，"bright" 表示亮目标，"dark" 表示暗目标
    """
    direct_threshold, _ = otsu_threshold(image)
    direct_binary = threshold_segment(image, direct_threshold, foreground=foreground)

    weighted_hist, gradient = compute_gradient_weighted_histogram(image)
    transformed_threshold, _ = otsu_threshold_from_histogram(weighted_hist)
    transformed_binary = threshold_segment(image, transformed_threshold, foreground=foreground)

    original_hist = compute_histogram(image).astype(np.float64)
    gray_levels = np.arange(256)

    plt.figure(figsize=(15, 8))

    plt.subplot(2, 3, 1)
    plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    plt.title(f"{title} 原图")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(normalize_for_display(gradient), cmap="gray", vmin=0, vmax=1)
    plt.title("Sobel 梯度幅值图")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(direct_binary, cmap="gray", vmin=0, vmax=1)
    plt.title(f"直接 Otsu 分割 T={direct_threshold}")
    plt.axis("off")

    plt.subplot(2, 3, 4)
    plt.bar(gray_levels, original_hist, width=1.0, color="#9aa6b2", alpha=0.85)
    plt.axvline(direct_threshold, color="#dc2626", linewidth=2.5, label=f"直接 T={direct_threshold}")
    plt.title("原始灰度直方图")
    plt.xlabel("灰度值")
    plt.ylabel("像素数量")
    plt.legend()

    plt.subplot(2, 3, 5)
    plt.bar(gray_levels, weighted_hist, width=1.0, color="#60a5fa", alpha=0.85)
    plt.axvline(transformed_threshold, color="#7c2d12", linewidth=2.5, label=f"修正后 T={transformed_threshold}")
    plt.title("平均梯度加权直方图")
    plt.xlabel("灰度值")
    plt.ylabel("平均梯度幅值")
    plt.legend()

    plt.subplot(2, 3, 6)
    plt.imshow(transformed_binary, cmap="gray", vmin=0, vmax=1)
    plt.title(f"直方图修正后 Otsu 分割 T={transformed_threshold}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"{title}")
    print(f"网络图像地址：{NETWORK_IMAGE_URL}")
    print(f"直接 Otsu 阈值：{direct_threshold}")
    print(f"直方图修正后 Otsu 阈值：{transformed_threshold}")
    print("-" * 40)


network_gray_image = load_grayscale_image_from_url(NETWORK_IMAGE_URL)

compare_direct_and_transformed_otsu(
    network_gray_image,
    title="网络硬币灰度图像",
    foreground="bright",
)

### 11.1 结果观察

观察上面的网络图像实验结果时，可以重点比较：

1. 网络图像的 **原始直方图** 与 **平均梯度加权直方图** 的形状是否发生变化。
2. 直方图修正后得到的 Otsu 阈值是否与直接 Otsu 阈值不同。
3. 直接 Otsu 主要依据整幅图像的灰度分布选择阈值，容易受到大面积背景灰度分布的影响。
4. 平均梯度加权直方图更强调目标与背景交界处的灰度级，因此修正后的阈值会更多体现边界信息。
5. 比较两幅二值分割结果，观察目标边界、背景误分区域和细小目标是否发生变化。

需要注意：直方图变换法并不是万能的。它依赖梯度信息，如果图像噪声很强，平均梯度加权直方图也可能把噪声边缘放大。因此在实际应用中，常常需要结合平滑滤波、局部阈值或形态学后处理一起使用。

## 12. 实验小结

本实验使用程序构造的经典硬币灰度测试图像和光照不均灰度测试图像，实现了基于 Otsu 大津法的自动阈值分割，并比较了全局 Otsu、分块局部 Otsu 以及直方图变换后 Otsu。

需要掌握的重点：

1. Otsu 方法通过最大化类间方差自动选择阈值。
2. 灰度直方图能够帮助观察前景和背景灰度分布。
3. 全局 Otsu 对整幅图像使用一个阈值，适合光照较均匀、前景背景灰度差异明显的图像。
4. 当同一背景区域在不同位置灰度差异很大时，全局阈值容易把暗背景或亮背景误分。
5. 局部 Otsu 对每个块单独计算阈值，可以适应局部光照变化。
6. 局部 Otsu 的分块大小会影响分割结果，过小可能产生不稳定阈值，过大则接近全局分割。
7. 直方图变换法可以利用梯度等局部性质修正直方图，使阈值选择过程更关注目标边界信息。
8. 在实际图像中，需要结合光照条件、目标大小和噪声情况选择合适的阈值分割方法。